# Import

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq 
from mapd.sinq_builders import build_composite_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Load learners vs. control sinq

In [ ]:
sinq_0 = Sinq(sinqname='Figure1')
sinq_1 = Sinq(sinqname='Fig1_control')
sinq = build_composite_sinq(name='Figure2_learn_v_no_learn_lda',sources=[sinq_0,sinq_1])

In [ ]:
op_df = sinq.display_df(show_tags=True)
op_df['learn'] = sinq.has_tag('learn')
op_df['control'] = sinq.has_tag('control')
op_df[['genotype','tags','learn','control']]

# Compare learners vs. not

In [ ]:
# print([i for i in op_df['genotype'].unique()])
list(op_df['genotype'].unique())

In [ ]:
geno_map = {'Hot-Cell-Gal4 (test)':             'HC',
 'Hot-Cell-LexA_Chr;81A06_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;35c09_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;35C09_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;78E05_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;31H05_pJFRC7': 'HC',
 'SS61350_pJFRC7':                  '+',
 '+;31H05-Gal4_pJFRC7':             '+',
 '+;TH-Gal4_UAS-Kir2_1':            '+;TH-Gal4_UAS-Kir2_1',
 '+;pJFRC7;31H05-Gal4':             '+',
 '+;pJFRC7;SS61350':                '+',
 'ppk-Gal4;10XUAS-ChR in WT':       '+;ppk-GAL4;10XUAS-ChR',
 '+;ppk-GAL4;10XUAS-ChR':           '+;ppk-GAL4;10XUAS-ChR',
 'Hot-Cell-Gal4_ChrinWT':           'HC',
 'w, NorpA':                        'NorpA',
 '+;pJFRC7;+':                      '+',
 'norpA;cry':                       '+',
 '+;pJFRC7;35C09-GAL4':             '+',
 'ss61350_pJFRC7':                  '+',
 'norpAE55':                        'NorpA',
 '+;pFJRC7;+':                      '+',
 'norpA':                           'NorpA',
 '+;pJFRC7;81A06-GAL4':             '+',
}

op_df['geno_smpl'] = op_df['genotype'].map(geno_map)
op_df.loc[op_df['geno_smpl'].isna()]

## Additional factors

In [ ]:
# op_df.loc['250506_F2_C1','learn'] = True
# op_df.loc['250506_F2_C1','learn_note'] = 'ok learner, got frustrated a bit'

# op_df.loc['250514_F1_C1','learn'] = True
# op_df.loc['250514_F1_C1','learn_note'] = 'ok learner, timed out a bit'

# op_df.loc['251219_F1_C2','learn'] = True
# op_df.loc['251219_F1_C2','learn_note'] = 'easy targets'

# # Don't learn for some reason

# op_df.loc['250304_F2_C1','learn'] = False
# op_df.loc['250304_F2_C1','learn_note'] = 'green, frustrated'

# op_df.loc['210331_F2_C1','learn'] = False
# op_df.loc['210331_F2_C1','learn_note'] = 'Not sure...'

# op_df.loc['250226_F2_C1','learn'] = False
# op_df.loc['250226_F2_C1','learn_note'] = 'Never got it, tried hard'

# op_df.loc['250228_F1_C1','learn'] = False
# op_df.loc['250228_F1_C1','learn_note'] = 'Adjusted baseline, but never got it'

# op_df.loc['250923_F1_C1','learn'] = False
# op_df.loc['250923_F1_C1','learn_note'] = 'learns late?'

# op_df.loc['250924_F1_C1','learn'] = False
# op_df.loc['250924_F1_C1','learn_note'] = 'tires out'

# op_df.loc['250925_F2_C1','learn'] = False
# op_df.loc['250925_F2_C1','learn_note'] = 'tires out'

op_df.loc[op_df['learn'], 'learn_class'] = 0  # Control
op_df.loc[op_df['control'], 'learn_class'] = 1  # Control
op_df.loc[(op_df['control'] == op_df['learn']), 'learn_class'] = 2  # borderline cases

In [ ]:
op_df['success_rate'] =         op_df['successes'] / op_df['num_trials']
op_df['hard_success_rate'] =    op_df['hard_successes'] / op_df['num_trials']

op_df['as_off_over_successes'] =        np.log10(op_df['outcome_fractions_as_off'] * op_df['num_trials'] + 1) / (op_df['successes'] + 1)
op_df['as_off_over_hard_successes'] =   np.log10(op_df['outcome_fractions_as_off'] * op_df['num_trials'] + 1) / (op_df['hard_successes'] + 1)

op_df['rms_velocity_bar'] = op_df['rms_velocity'] / op_df['duration']
op_df['holding_cost_bar'] = op_df['holding_cost'] / op_df['duration']
op_df['probe_positive_effort_bar'] = op_df['probe_positive_effort'] / op_df['duration']

In [ ]:
op_df

# Compare factors, learners vs control

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_factor_by_learners(df: pd.DataFrame, factor_col: str, group_col: str = "learn",
                            fig_dir: str = "./Figure2/factor_plots",
                            title: str | None = None, jitter: float = 0.18, seed: int = 0):
    """
    Compare distributions between learners (True) vs non-learners (False).
    Left: jittered strip plot of all points
    Right: box plot with median, mean, ±SD, and overlaid full-range whiskers (min→max).
    Hover shows row index + value.
    """
    # Clean and order groups
    dfx = df[[group_col, factor_col, "tags"]].copy()
    dfx = dfx.dropna(subset=[group_col, factor_col])
    dfx["_index"] = dfx.index  # keep original index
    dfx["_grp_bool"] = dfx[group_col].astype(bool)
    dfx["_grp_label"] = np.where(dfx["_grp_bool"], "Learners","Control",)
    dfx['_tags'] = f"{', '.join(dfx['tags'])}"
    def format_tags(tags):
        if isinstance(tags, (list, tuple)) and len(tags) > 0:
            return ", ".join(tags)
        return ""

    dfx["_tags_str"] = dfx["tags"].apply(format_tags)

    order = ["Control", "Learners"]

    # Colors
    try:
        import seaborn as sns, matplotlib.colors as mcolors
        palette = sns.color_palette("Set2", 2)
        colors = {order[0]: mcolors.to_hex(palette[0]), order[1]: mcolors.to_hex(palette[1])}
    except Exception:
        colors = {order[0]: "#1f77b4", order[1]: "#2ca02c"}

    # Summary stats
    stats = (
        dfx.groupby("_grp_label")[factor_col]
           .agg(["count", "mean", "std", "median", "min", "max"])
           .reindex(order)
    )

    # Build figure with two panels sharing Y
    fig = make_subplots(
        rows=1, cols=2, shared_yaxes=True,
        column_widths=[0.55, 0.45],
        horizontal_spacing=0.08,
        subplot_titles=("All points", "Box + mean ± SD + full-range whiskers")
    )

    # --- Left panel: jittered strip of all points ---
    rng = np.random.RandomState(seed)
    for i, label in enumerate(order):
        sub = dfx[dfx["_grp_label"] == label]
        sub = dfx[dfx["_grp_label"] == label]
        xj = i + rng.uniform(-jitter, jitter, size=len(sub))
        fig.add_trace(
            go.Scatter(
                x=xj, y=sub[factor_col],
                mode="markers",
                name=label,
                legendgroup=label,
                marker=dict(size=7, opacity=0.6, color=colors[label], line=dict(width=0)),
                hovertemplate=(
                    f"{label}"
                    "<br>index: %{customdata[0]}"
                    f"<br>{factor_col}: %{{y:.4g}}"
                    "<br>tags: %{customdata[1]}"
                    "<extra></extra>"
                ),
                customdata = list(zip(sub["_index"].astype(str),
                        sub["_tags_str"].astype(str),
                    )
                ),
                showlegend=True
            ),
            row=1, col=1
        )

    # Cosmetic x-axis for left panel
    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(order))),
        ticktext=order,
        row=1, col=1
    )

    # --- Right panel: box plot + mean ± SD + full range whiskers ---
    for label in order:
        sub = dfx[dfx["_grp_label"] == label]
        fig.add_trace(
            go.Box(
                y=sub[factor_col],
                x=[label] * len(sub),
                name=label,
                legendgroup=label,
                marker_color=colors[label],
                boxmean="sd",           # mean line + ±SD band
                boxpoints="all",        # show points
                jitter=0.4,
                pointpos=0,
                customdata = list(zip(sub["_index"].astype(str),
                        sub["_tags_str"].astype(str),
                    )
                ),
                hovertemplate=(
                    f"{label}"
                    "<br>index: %{customdata[0]}"
                    f"<br>{factor_col}: %{{y:.4g}}"
                    "<br>tags: %{customdata[1]}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=2
        )

    # Add true min→max whisker lines on the box panel
    shapes = []
    for label in order:
        mn = stats.loc[label, "min"]
        mx = stats.loc[label, "max"]
        shapes.append(dict(
            type="line", xref="x2", yref="y2",
            x0=label, x1=label, y0=mn, y1=mx,
            line=dict(width=2, color="rgba(0,0,0,0.5)")
        ))
    fig.update_layout(shapes=shapes)

    # Titles and layout
    if title is None:
        title = f"Distribution of {factor_col}: Learners vs Non-learners"
    fig.update_layout(
        title=title,
        width=1100, height=700,
        margin=dict(l=60, r=60, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hoverlabel=dict(font_size=14)
    )
    fig.update_yaxes(title_text=factor_col, row=1, col=1)
    fig.update_xaxes(title_text="", row=1, col=2)

    fig.write_image(f'{fig_dir}/learn_v_control_{factor_col}.svg')
    return fig

# Example:
# fig = plot_factor_by_learners(proj_df, "your_factor_column")
# fig.show()

In [ ]:
op_df

In [ ]:
fig = plot_factor_by_learners(op_df, factor_col='hi_state_on_target',group_col='learn')
fig.show()
# hi_state_on_target               0.622351
# outcome_fractions_no_as_no_mv    0.569569
# hard_success_rate                0.244791
# as_off_over_hard_successes      -0.050685
# success_rate                    -0.320152
# hi_lo_shift                     -0.351116
# dtype: float64

## PCA on all factors

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

folder_path = "./Figure2/exports"
figure_path = "./Figure2/"

labels = ['learn_class','learn','note_text','geno_smpl']

factors = [
    'outcome_fractions_no_as_no_mv',
    'outcome_fractions_no_as_mv',  
    'outcome_fractions_as_off',
    'outcome_fractions_as_off_late',
    'outcome_fractions_timeout',
    'hi_lo_shift',
    'lo_state_on_target',
    'hi_state_on_target',
    'lo_target_off_state',
    'hi_target_off_state',
    'success_rate',
    'hard_success_rate',
    'as_off_over_successes',
    'as_off_over_hard_successes',
    'rms_velocity_bar',
    'holding_cost_bar',
    'probe_positive_effort_bar',
]
pca_df = op_df.loc[:,labels+factors]

# y = op_df['learners']
X = pca_df.drop(columns=labels)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X,)

pca = PCA(n_components=3,
          copy=True, 
          whiten=False, 
          svd_solver='full', 
          tol=0.0, 
          iterated_power='auto')
X_pca = pca.fit_transform(x_scaled)


# Make a projection DataFrame
proj_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3"],
    index=pca_df.index
)

proj_df[labels] = pca_df[labels]

In [ ]:
col_str = 'learn_class'

cats = pd.Index(sorted(proj_df[col_str].unique()))
n = len(cats)
palette = sns.color_palette("tab10", n)
color_map = dict(zip(cats, palette))

# Reuse proj_df and color_map from above
plot_df = proj_df.reset_index().rename(columns={"index":"fly_id"})

# Convert seaborn/matplotlib colors to hex for plotly
color_map = dict(zip(cats, palette))
color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

fig = px.scatter_3d(
    plot_df,
    x="PC1", y="PC2", z="PC3",
    color=col_str,
    color_discrete_map=color_map_hex,
    hover_data=["fly_id", "geno_smpl",'note_text'],
    title="PCA Projections (PC1–PC3)"
)
fig.update_traces(marker=dict(size=5, opacity=0.9))
fig.update_layout(
    legend_title_text="Genotype",
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
    )
)

fig.update_layout(
    width=1100, height=850,                    # << size here
    margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
    font=dict(size=16),
    legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
        xaxis=dict(tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=12)),
        zaxis=dict(tickfont=dict(size=12)),
    ),
    hoverlabel=dict(font_size=14)
)

fig.show()

In [ ]:
for leg in [False,True]:
    for col in ['learn','geno_smpl']:
        cats = pd.Index(sorted(proj_df[col].unique()))
        n = len(cats)
        palette = sns.color_palette("tab20", n) if n <= 20 else sns.husl_palette(n)
        color_map = dict(zip(cats, palette))
        point_colors = [mcolors.to_hex(color_map[g]) for g in proj_df[col]]

        # --- Helper to save a 2D scatter as SVG ---
        def save_pc2d(proj, pca_obj, xpc="PC1", ypc="PC2", fname="plot.svg"):
            fig, ax = plt.subplots(figsize=(7.5, 6))
            ax.scatter(proj[xpc], proj[ypc], s=50, alpha=0.9, c=point_colors)

            # Axis labels with variance explained
            pc_names = ["PC1","PC2","PC3"]
            exp = {pc_names[i]: pca_obj.explained_variance_ratio_[i] for i in range(3)}
            ax.set_xlabel(f"{xpc} ({exp[xpc]:.1%} var)")
            ax.set_ylabel(f"{ypc} ({exp[ypc]:.1%} var)")
            ax.set_title(f"PCA: {ypc} vs {xpc}")

            # Legend (handles many genotypes)
            handles = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=7,
                                markerfacecolor=mcolors.to_hex(color_map[g])) for g in cats]
            ncol = 1 if n < 12 else 2 if n <= 24 else 3
            if leg:
                ax.legend(handles, list(cats), title=col,
                        bbox_to_anchor=(1.02, 1), loc="upper left", ncol=ncol, fontsize=8)

            ax.grid(True, alpha=0.2)
            plt.tight_layout()
            fig.savefig(fname, format="svg", dpi=300, bbox_inches="tight")
            plt.close(fig)
            print(f"Saved {fname}")

        # --- Make the two figures ---
        save_pc2d(proj_df, pca, xpc="PC1", ypc="PC2", fname=f"./Figure2/pca_PC2_vs_PC1_{col}_legend_{leg}.svg")
        save_pc2d(proj_df, pca, xpc="PC1", ypc="PC3", fname=f"./Figure2/pca_PC3_vs_PC1_{col}_legend_{leg}.svg")


In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=X.columns
)

# Optional: variance-scaled loadings (correlations) sometimes used for “importance”:
# These are eigenvectors scaled by sqrt(eigenvalues)
var_scaled_loadings = pd.DataFrame(
    (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=X.columns
)

# --- 4) Plot PC1 loadings sorted by absolute magnitude ---
for pc in ["PC1",'PC2','PC3']:
    pc_l = loadings.loc[pc].sort_values(key=np.abs, ascending=False)

    fig = plt.figure(figsize=(10, 5))
    pc_l.plot(kind="bar")
    plt.title(f"{pc} Loadings (feature weights)")
    plt.ylabel("Loading")
    plt.xlabel("Features (sorted by |loading|)")
    plt.tight_layout()

    # If you want the top-N features programmatically:
    topN = 15
    # pc1_top = pc_l.head(topN)
    # print(pc1_top)
    fig.savefig(f'./Figure2/{pc}.svg')

# PCA on a subset of factors

In [ ]:
folder_path = "./Figure2/exports"
figure_path = "./Figure2/"

labels = ['geno_smpl','learn_class','learn','note_text']

factors = [
    'outcome_fractions_no_as_no_mv',        # pc1, 3
    #'outcome_fractions_no_as_mv',          # pc3, 2
    'outcome_fractions_as_off',            # pc2, 1
    #'outcome_fractions_as_off_late',
    #'outcome_fractions_timeout',           # pc2, 4
    #'hi_lo_shift',
    'lo_state_on_target',                   # pc1, 2
    'hi_state_on_target',                   # pc1, 1
    'lo_target_off_state',                  # pc3, 1
    'hi_target_off_state',                  # pc3, 4
    'success_rate',                         # pc2, 5
    'hard_success_rate',                    # pc1, 4
    'as_off_over_successes',                # pc3, 4
    'as_off_over_hard_successes',           # pc1, 5
    'rms_velocity_bar',                     # pc2, 3
    #'holding_cost_bar',
    'probe_positive_effort_bar',            # pc2, 2, pc3, 5
]
pca_df = op_df.loc[:,labels+factors]

# y = op_df['learners']
X = pca_df.drop(columns=labels)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X,)

pca = PCA(n_components=3,
          copy=True, 
          whiten=False, 
          svd_solver='full', 
          tol=0.0, 
          iterated_power='auto')
X_pca = pca.fit_transform(x_scaled)


# Make a projection DataFrame
proj_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3"],
    index=pca_df.index
)

proj_df[labels] = pca_df[labels]

In [ ]:
import plotly.io as pio
from IPython.display import display


col_str = 'learn'

cats = pd.Index(sorted(proj_df[col_str].unique()))
n = len(cats)
palette = sns.color_palette("tab10", n)
color_map = dict(zip(cats, palette))

# Reuse proj_df and color_map from above
plot_df = proj_df.reset_index().rename(columns={"index":"fly_id"})

# Convert seaborn/matplotlib colors to hex for plotly
color_map = dict(zip(cats, palette))
color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

fig = px.scatter_3d(
    plot_df,
    x="PC1", y="PC2", z="PC3",
    color=col_str,
    color_discrete_map=color_map_hex,
    hover_data=["fly_id", "geno_smpl",'note_text'],
    title="PCA Projections (PC1–PC3)"
)
fig.update_traces(marker=dict(size=5, opacity=0.9))
fig.update_layout(
    legend_title_text=col_str,
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
    )
)

fig.update_layout(
    width=1100, height=850,                    # << size here
    margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
    font=dict(family="Arial",size=16),
    legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
        xaxis=dict(tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=12)),
        zaxis=dict(tickfont=dict(size=12)),
    ),
    hoverlabel=dict(font_size=14)
)


fig.update_layout(
    scene_camera=dict(
        eye=dict(x=-1.6, y=-2.1, z=1),   # rotate around X/Y/Z
        up=dict(x=0, y=0, z=1.5)       # z-axis is vertical
    )
)

# fig = go.FigureWidget(fig)

# show the fig and attach listener
display(fig)

# Attach a listener that prints the camera dict when you rotate
from IPython.display import Javascript

display(Javascript("""
require(["base/js/namespace"], function(Jupyter) {
  Jupyter.notebook.kernel.execute("last_camera = None")  // initialize
  const figs = document.querySelectorAll('.js-plotly-plot');
  figs.forEach(fig => {
    fig.on('plotly_relayout', ev => {
      if(ev['scene.camera']){
        const cam = JSON.stringify(ev['scene.camera']);
        Jupyter.notebook.kernel.execute("last_camera = " + cam);
        console.log("Camera updated:", ev['scene.camera']);
      }
    });
  });
});
"""))

# fig.show()
# fig.write_image(f'./Figure3/pca_proj_3d.png)
# fig.write_image("pca_projection.png", width=4400, height=3400, scale=1)

In [ ]:
for leg in [False,True]:
    for col in ['learn','geno_smpl']:
        cats = pd.Index(sorted(proj_df[col].unique()))
        n = len(cats)
        palette = sns.color_palette("tab20", n) if n <= 20 else sns.husl_palette(n)
        color_map = dict(zip(cats, palette))
        point_colors = [mcolors.to_hex(color_map[g]) for g in proj_df[col]]

        # --- Helper to save a 2D scatter as SVG ---
        def save_pc2d(proj, pca_obj, xpc="PC1", ypc="PC2", fname="plot.svg"):
            fig, ax = plt.subplots(figsize=(7.5, 6))
            ax.scatter(proj[xpc], proj[ypc], s=50, alpha=0.9, c=point_colors)

            # Axis labels with variance explained
            pc_names = ["PC1","PC2","PC3"]
            exp = {pc_names[i]: pca_obj.explained_variance_ratio_[i] for i in range(3)}
            ax.set_xlabel(f"{xpc} ({exp[xpc]:.1%} var)")
            ax.set_ylabel(f"{ypc} ({exp[ypc]:.1%} var)")
            ax.set_title(f"PCA: {ypc} vs {xpc}")

            # Legend (handles many genotypes)
            handles = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=7,
                                markerfacecolor=mcolors.to_hex(color_map[g])) for g in cats]
            ncol = 1 if n < 12 else 2 if n <= 24 else 3
            if leg:
                ax.legend(handles, list(cats), title=col,
                        bbox_to_anchor=(1.02, 1), loc="upper left", ncol=ncol, fontsize=8)

            ax.grid(True, alpha=0.2)
            plt.tight_layout()
            fig.savefig(fname, format="svg", dpi=300, bbox_inches="tight")
            plt.close(fig)
            print(f"Saved {fname}")

        # --- Make the two figures ---
        save_pc2d(proj_df, pca, xpc="PC1", ypc="PC2", fname=f"./Figure2/pca_PC2_vs_PC1_smpl_{col}_legend_{leg}.png")
        save_pc2d(proj_df, pca, xpc="PC1", ypc="PC3", fname=f"./Figure2/pca_PC3_vs_PC1_smpl_{col}_legend_{leg}.png")


In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=X.columns
)

# Optional: variance-scaled loadings (correlations) sometimes used for “importance”:
# These are eigenvectors scaled by sqrt(eigenvalues)
var_scaled_loadings = pd.DataFrame(
    (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=X.columns
)

# --- 4) Plot PC1 loadings sorted by absolute magnitude ---
for pc in ["PC1",'PC2','PC3']:
    pc_l = loadings.loc[pc].sort_values(key=np.abs, ascending=False)

    fig = plt.figure(figsize=(10, 5))
    pc_l.plot(kind="bar")
    plt.title(f"{pc} Loadings (feature weights)")
    plt.ylabel("Loading")
    plt.xlabel("Features (sorted by |loading|)")
    plt.tight_layout()

    # If you want the top-N features programmatically:
    topN = 15
    # pc1_top = pc_l.head(topN)
    # print(pc1_top)
    fig.savefig(f'./Figure2/{pc}_simplified.svg')

# LDA on the above factors

In [ ]:
lda_df = op_df.loc[op_df['learn_class']<2]
lda_df['learn_class']

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
 
labels = ['geno_smpl','learn_class','learn','note_text']

factors = [
    'outcome_fractions_no_as_no_mv',        # pc1, 3
    #'outcome_fractions_no_as_mv',          # pc3, 2
    'outcome_fractions_as_off',            # pc2, 1
    #'outcome_fractions_as_off_late',
    #'outcome_fractions_timeout',           # pc2, 4
    #'hi_lo_shift',
    'lo_state_on_target',                   # pc1, 2
    'hi_state_on_target',                   # pc1, 1
    'lo_target_off_state',                  # pc3, 1
    'hi_target_off_state',                  # pc3, 4
    'success_rate',                         # pc2, 5
    'hard_success_rate',                    # pc1, 4
    'as_off_over_successes',                # pc3, 4
    'as_off_over_hard_successes',           # pc1, 5
    'rms_velocity_bar',                     # pc2, 3
    #'holding_cost_bar',
    'probe_positive_effort_bar',            # pc2, 2, pc3, 5
]

X = lda_df[factors].values
y = lda_df['learn'].astype(int).values

X_ = op_df[factors].values
y_ = op_df['learn_class'].astype(int).values

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X,)
x_scaled_ = scaler.fit_transform(X_,)

pca = PCA(n_components=3,
          copy=True, 
          whiten=False, 
          svd_solver='full', 
          tol=0.0, 
          iterated_power='auto')
Z = pca.fit_transform(x_scaled)
Z_ = pca.fit_transform(x_scaled_)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, roc_auc_score

# --- 1) Fit LDA in the PCA space
lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
lda.fit(Z, y)

# Binary LDA => one row in coef_
w_pc = lda.coef_.ravel()                 # direction in PC space (not unit)
b_pc = lda.intercept_.item()             # bias in PC space
w_pc_unit = w_pc / np.linalg.norm(w_pc)  # unit vector (direction only)

# Discriminant scores and predictions using the true linear form
scores = Z @ w_pc + b_pc                 # same as lda.decision_function(Z)
y_pred = (scores >= 0).astype(int)

print("Acc:", accuracy_score(y, y_pred))
print("ROC-AUC:", roc_auc_score(y, scores))

# --- 2) (Optional) Project onto the *unit* axis if you prefer a centered 1D view
scores_unit = Z @ w_pc_unit
# Class means along the unit axis (for plotting/intuition)
mu0 = scores_unit[y==0].mean()
mu1 = scores_unit[y==1].mean()

# --- 3) Map direction back to original features
# PCA components_: shape (n_pcs, n_features); rows are PC loadings
V = pca.components_                      # (k, p) here k=3, p=len(factors)
# Direction in original (scaled) feature space:
w_orig_scaled = V.T @ w_pc_unit          # (p,)

# If you want *raw units*, undo the StandardScaler scaling
w_orig_raw = w_orig_scaled / scaler.scale_
w_orig_raw /= np.linalg.norm(w_orig_raw) # unit vector in raw units

# Nice, readable series
w_scaled_s = pd.Series(w_orig_scaled, index=factors).sort_values(ascending=False)
w_raw_s    = pd.Series(w_orig_raw,    index=factors).sort_values(ascending=False)

print("\nTop drivers (scaled feature space):")
display(w_scaled_s.head(10))
print("\nTop drivers (raw units):")
display(w_raw_s.head(10))

# --- 4) Handy objects you might reuse
lda_results = {
    "w_pc": w_pc,               # discriminant direction (PC space)
    "b_pc": b_pc,               # intercept
    "w_pc_unit": w_pc_unit,     # unit direction (PC space)
    "scores": scores,           # linear discriminant scores (w·z + b)
    "scores_unit": scores_unit, # projection on unit axis (no intercept)
    "y_pred": y_pred,
    "w_orig_scaled": w_orig_scaled,  # feature contributions (scaled space)
    "w_orig_raw": w_orig_raw,        # feature contributions (raw units)
}


In [ ]:
ftrfig = Figure(figsize=(4,2))
# plt.figure(figsize=(5,5))
ftrax = ftrfig.add_subplot(1, 1, 1)

ftrax = w_raw_s.plot(kind="barh", ax=ftrax)
ftrax.set_title("Feature contributions toward the '1' class (raw units)")
ftrax.set_xlabel("Loading along LDA direction")
ftrax.invert_yaxis()
ftrfig.tight_layout()
display(ftrfig)

In [ ]:
ftrfig = Figure(figsize=(4,2))
# plt.figure(figsize=(5,5))
ftrax = ftrfig.add_subplot(1, 1, 1)

ftrax = w_scaled_s.plot(kind="barh", ax=ftrax)
ftrax.set_title("Feature contributions toward the '1' class (scaled units)")
ftrax.set_xlabel("Loading along LDA direction")
ftrax.invert_yaxis()
ftrfig.tight_layout()
display(ftrfig)
ftrfig.savefig(fname='./Figure2/LDA_w_scaled.svg',format='svg')

In [ ]:
tags = sinq.all_tags()
print(tags)

In [ ]:
expts = [
    '1s_stim',
    'replay',
    'norpa',
    ['norpa','no_antenna'],
    'easy_targets',
    'no_antenna',
    'vnc_cut'
]

In [ ]:
# def _get_gstr(g):
#     g = g.replace('.', '_')
#     g = g.replace('>', '_')
#     g = g.replace('/', '_')
#     g = g.replace('[', '')
#     g = g.replace(']', '')
#     g = g.replace('*', '')
#     return g

# # pc12 = Z[:, :2]
# pc12 = Z_[:, :2]
# origin = pc12.mean(axis=0)
# # w2 = w_pc_unit[:2]
# w2 = w_pc[:2]

# op_df["PC1"] = pc12[:, 0]
# op_df["PC2"] = pc12[:, 1]

# op_df["_hover"] = (
#     "index: " + plot_df.index.astype(str)
#     + "<br>tags: " + plot_df["_tags_str"]
# )

# for expt in expts:
#     LDA_fig = Figure(figsize=(8,9))
#     # plt.figure(figsize=(5,5))
#     ax1 = LDA_fig.add_subplot(1, 1, 1)

#     ax1.scatter(pc12[:,0], pc12[:,1], c=y_, s=40, alpha=0.7, cmap="coolwarm")

#     hi_contrast = "#FFB000"   # great against blue/cool colors
    
#     labels = sinq.rows_with_tags(expt)
#     print(f'Plotting {len(labels)} flies in {expt} experiments')
#     sel = sinq.df.index.get_indexer(labels)

#     ax1.scatter(pc0[sel,0], pc0[sel,1],
#                 s=28, c=hi_contrast, edgecolors='none',
#                 marker='o', zorder=5)

#     # direction arrow (scaled for visibility)
#     scale = 2.0 * pc12.std(0).mean()
#     ax1.arrow(origin[0], origin[1], scale*w2[0], scale*w2[1], 
#             length_includes_head=True, head_width=0.06*scale)

#     # decision boundary: {z : w·z + b = 0} → line with normal w2
#     # pick a point x0 on the boundary in 2D by solving w2·x0 + (w_pc[2:]*mean(others) + b) ≈ 0
#     # Since we're only plotting (PC1, PC2), treat other PCs at their means (≈0 after PCA).
#     b2 = b_pc  # others ~ 0 in Z, so intercept stays b
#     # Point on line: x0 = -b2 * w2 / ||w2||^2
#     x0 = -b2 * w2 / (np.linalg.norm(w2)**2 + 1e-12)
#     # Direction along the line is perpendicular to w2:
#     v = np.array([-w2[1], w2[0]])
#     t = np.linspace(-2, 2, 200)
#     line = x0[None,:] + t[:,None]*v[None,:]
#     ax1.plot(line[:,0], line[:,1], '--', lw=2)

#     ax1.set_aspect('equal', adjustable='datalim')
#     ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2'); ax1.set_title('LDA in PCA space')
#     if isinstance(expt, str):
#         es = expt
#     elif isinstance(expt,list):
#         es = '_'.join(expt)
#     LDA_fig.savefig(f'./Figure2/genos_in_LDA_space/LDA_in_PCA_space_{es}.svg')
#     LDA_fig.savefig(f'./Figure2/genos_in_LDA_space/LDA_in_PCA_space_{es}.png')


# display(LDA_fig)


In [ ]:
for pc2idx in [1,2]:
    pc12 = Z_[:,[0,pc2idx]]
    op_df["PC1"] = Z_[:,0]
    op_df["PC2"] = Z_[:,pc2idx]

    origin = pc12.mean(axis=0)
    # w2 = w_pc_unit[:2]
    w2 = w_pc[[0,2]]

    scale = 2.0 * pc12.std(0).mean()

    # decision boundary: {z : w·z + b = 0} → line with normal w2
    # pick a point x0 on the boundary in 2D by solving w2·x0 + (w_pc[2:]*mean(others) + b) ≈ 0
    # Since we're only plotting (PC1, PC2), treat other PCs at their means (≈0 after PCA).
    b2 = b_pc  # others ~ 0 in Z, so intercept stays b
    # Point on line: x0 = -b2 * w2 / ||w2||^2
    x0 = -b2 * w2 / (np.linalg.norm(w2)**2 + 1e-12)
    # Direction along the line is perpendicular to w2:
    v = np.array([-w2[1], w2[0]])
    t = np.linspace(-2, 2, 200)
    line = x0[None,:] + t[:,None]*v[None,:]

    op_df["_tags_str"] = op_df["tags"].apply(
        lambda t: ", ".join([t]) if t else ""
    )

    op_df["_hover"] = (
        "index: " + op_df.index.astype(str)
        + "<br>tags: " + op_df["_tags_str"]
    )

    import plotly.graph_objects as go

    color_map = {
        0: "#000000",   # blue
        1: "#989898",   # orange
        2: "#484848",   # orange
    }

    for expt in expts:
        
        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=op_df["PC1"],
                y=op_df["PC2"],
                mode="markers",
                marker=dict(
                    size=8,
                    color=op_df["learn_class"].map(color_map),
                    colorscale="RdBu",
                    opacity=0.7,
                ),
                hovertext=op_df["_hover"],
                hoverinfo="text",
                name="All points",
            )
        )

        labels = sinq.rows_with_tags(expt)
        print(f'Plotting {len(labels)} flies in {expt} experiments')
        sel_df = op_df.loc[labels]

        fig.add_trace(
            go.Scatter(
                x=sel_df["PC1"],
                y=sel_df["PC2"],
                mode="markers",
                marker=dict(
                    size=9,
                    color="#FFB000",
                    opacity=0.95,
                ),
                hovertext=sel_df["_hover"],
                hoverinfo="text",
                name=f"tagged: {expt}",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[origin[0], origin[0] + scale * w2[0]],
                y=[origin[1], origin[1] + scale * w2[1]],
                mode="lines+markers",
                line=dict(width=3),
                name="LDA direction",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=line[:, 0],
                y=line[:, 1],
                mode="lines",
                line=dict(dash="dash", width=2),
                name="decision boundary",
            )
        )
        fig.update_layout(
            title="LDA in PCA space",
            xaxis_title="PC1",
            yaxis_title="PC2",
            width=800,
            height=800,
            hovermode="closest",
        )

        fig.update_yaxes(scaleanchor="x", scaleratio=1)

        if isinstance(expt, str):
            es = expt
        elif isinstance(expt,list):
            es = '_'.join(expt)
        for fmt in ['html','svg','png']:
            fname = f"./Figure2/expts_in_LDA_space/{fmt}/LDA_in_PC1_PC{pc2idx+1}_{es}.{fmt}"
            if fmt=='html':
                fig.write_html(fname)
            else:
                fig.write_image(fname)

In [ ]:
display(fig)

In [ ]:
w_pc_unit

In [ ]:
sinq.rows_with_tags('replay')

In [ ]:
# from sklearn.metrics import roc_curve
# fpr, tpr, thr = roc_curve(y, scores)
# j = tpr - fpr
# threshold = thr[np.argmax(j)]  # Youden's J

# # Predicted class by the 1D rule
# y_pred = (scores >= threshold).astype(int)
# y_pred

In [ ]:
# scores = lda.decision_function(Z)  # = Z @ w_pc + b, 1D array
scores = lda.decision_function(Z_)  # = Z @ w_pc + b, 1D array
thr = 0.0

# fig, ax = plt.subplots(figsize=(5,3.2))
hstfig = Figure(figsize=(5,3.2))
# plt.figure(figsize=(5,5))
hstrax = hstfig.add_subplot(1, 1, 1)

for cls in (0, 1,2):
    # hstrax.hist(scores[y==cls], bins=16,  alpha=0.55, label=f"class {cls}")
    hstrax.hist(scores[y_==cls], bins=16,  alpha=0.55, label=f"class {cls}")

hstrax.axvline(thr, linestyle="--", linewidth=2)
hstrax.set_xlabel("LDA score  (w·z + b)")
hstrax.set_ylabel("Density")
hstrax.legend(frameon=False)
hstrax.set_title("Projection onto LDA direction with decision threshold")
hstfig.tight_layout()
display(hstfig)
hstfig.savefig('./Figure2/flies_onto_LDA.svg')

# UMAP

In [ ]:
# import umap

# umap_df = sinq.df.copy()   # Don't mess with sinq
# learner_types = ['SS61350_pJFRC7', '+;31H05-Gal4_pJFRC7', 'Hot-Cell']
# non_learner_types = ['iav_Kir2_1','+;TH-Gal4_UAS-Kir2_1	','iav-Gal4_+;+;UAS-Kir2_1','+;iav-Gal4_UAS-Kir2_1']

# geno_map = {'Hot-Cell-Gal4 (test)':'HC',
#     'Hot-Cell-LexA_Chr;81A06_pJFRC7':'HC',
#     'Hot-Cell-LexA_Chr;35c09_pJFRC7':'HC',
#     'Hot-Cell-LexA_Chr;35C09_pJFRC7':'HC',
#     'Hot-Cell-LexA_Chr;78E05_pJFRC7':'HC',
#     'Hot-Cell-LexA_Chr;31H05_pJFRC7':'HC',
#     'iav_Kir2_1':'w;iav>kir2.1',
#     '31H05_pJFRC7':'w',
#     'SS61350_pJFRC7':'+',
#     'iav-Gal4_+;+;UAS-Kir2_1':'+;iav>kir2.1',
#     '+;iav-Gal4_UAS-Kir2_1':'+;iav>kir2.1',
#     '+;31H05-Gal4_pJFRC7':'+',
#     '+;TH-Gal4_UAS-Kir2_1':'+;TH>kir2.1'}


# umap_df['learners'] = sinq.df['genotype'].apply(
#     lambda x: any(t in x for t in learner_types)
# )
# umap_df['geno_smpl'] = sinq.df['genotype'].map(geno_map)
# umap_df['rms_velocity_over_time'] = umap_df['rms_velocity'] / umap_df['duration']
# umap_df['norm_rms_velocity'] = umap_df['rms_velocity_over_time'] / umap_df['rms_velocity_over_time'].max()

# factors = [
#     'geno_smpl',
#     'learners',
#     'norm_rms_velocity',
#     'outcome_fractions_no_as_no_mv', 
#     'outcome_fractions_as_off',
#     # 'outcome_fractions_as_off_late',
#     # 'outcome_fractions_timeout',
#     'hi_lo_shift',
#     'lo_state_on_target',
#     'hi_state_on_target',
#     'lo_target_off_state',
#     'hi_target_off_state',
# ]

# umap_df = umap_df[factors]
# umap_df

# # Scaling
# # y = op_df['learners']
# X = umap_df.drop(columns=['learners','geno_smpl'])

# scaler = StandardScaler()
# x_scaled = scaler.fit_transform(X,)

# X_scaled = X.copy()
# X_scaled.loc[:,:] = x_scaled
# # syn_bin_L2_norm = syn_bin_df.copy()
# # syn_bin_L1_norm = syn_bin_df.copy()
# # for c in range(syn_bin_L2_norm.shape[1]):
# #     syn_bin_L2_norm.iloc[:,c] = syn_bin_L2_norm.iloc[:,c]/np.sqrt((syn_bin_L2_norm.iloc[:,c]**2).sum()) # L2 norm
# #     syn_bin_L1_norm.iloc[:,c] = syn_bin_L1_norm.iloc[:,c]/syn_bin_L1_norm.iloc[:,c].sum()

In [ ]:
# # Then run UMAP
# reducer = umap.UMAP(a=None, angular_rp_forest=False, b=None,
#      force_approximation_algorithm=False, init='spectral', learning_rate=1.0,
#      local_connectivity=1.0, low_memory=False, metric='euclidean',
#      metric_kwds=None, min_dist=.2, n_components=2, n_epochs=None,
#      n_neighbors=5, negative_sample_rate=5, output_metric='euclidean',
#      output_metric_kwds=None, random_state=42, repulsion_strength=1.0,
#      set_op_mix_ratio=1.0, spread=1.0, target_metric='categorical',
#      target_metric_kwds=None, target_n_neighbors=-1, target_weight=0.5,
#      transform_queue_size=4.0, transform_seed=42, unique=False, verbose=False)

# reducer.fit(X_scaled)
# embedding = reducer.transform(X_scaled)
# # Verify that the result of calling transform is
# # idenitical to accessing the embedding_ attribute
# assert(np.all(embedding == reducer.embedding_))
# print(embedding.shape)


In [ ]:
# cats = pd.Index(sorted(umap_df['geno_smpl'].unique()))
# n = len(cats)
# palette = sns.color_palette("Set2", n).as_hex()
# color_map = dict(zip(cats, palette))
# # point_colors = [mcolors.to_hex(color_map[g]) for g in proj_df[col]]
# umap_df['ctup'] = umap_df['geno_smpl'].map(color_map)

In [ ]:
# import plotly.express as px

# # Assume your index looks like yymmdd_FN_CM
# umap_df = umap_df.copy()
# umap_df["x"] = embedding[:, 0]
# umap_df["y"] = embedding[:, 1]

# fig = px.scatter(
#     umap_df,
#     x="x",
#     y="y",
#     color="geno_smpl",              # uses your categorical palette
#     hover_name=umap_df.index,       # shows the index on hover
#     hover_data={"geno_smpl": True}, # add other metadata if you want
#     color_discrete_map=color_map    # reuse your seaborn → hex map
# )

# fig.update_traces(marker=dict(size=8, opacity=0.8))
# fig.update_layout(
#     width=800,
#     height=600,
#     legend_title="Genotype"
# )

# fig.show()

# Extras

In [ ]:
sinq.df.loc['250226_F2_C1']

In [ ]:
T = sinq.restore_table('250226_F2_C1')

In [ ]:
T.extract_trial_properties()

In [ ]:
T.plot_outcomes(savefig=True)

# Functions

In [ ]:
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

import plotly.express as px
import matplotlib.colors as mcolors
import seaborn as sns
from collections import defaultdict

k_spring_constant = 0.0829 #uN/um

def probe_velocity(t,p):
        x = p.squeeze()
        t = t.squeeze()
        v = np.gradient(x, t)
        return v

def probe_rms_velocity(t,p):
    """Weights faster movements more"""
    v = probe_velocity(t,p)
    rms_vigor = np.sqrt(np.mean(v**2))
    return rms_vigor

# def probe_jerk_energy(self):
#     """A measure of effort in motor control"""
#     ds_time = self.time[self.downsample_probe]
#     jerk = np.gradient(self.probe_acceleration(), ds_time)
#     jerk_energy = np.sum(jerk**2) * np.diff(ds_time[2:3])
#     return jerk_energy

def probe_power(t,p):
    v = probe_velocity(t,p)
    power = k_spring_constant * p * v
    return power

def probe_work(t,p):
    """Work Done Against the Spring"""
    power = probe_power(t,p)
    work = np.trapezoid(power,t)
    return work

def probe_positive_effort(t,p):
    '''Assumes p is never negative'''
    vel = probe_velocity(t,p)
    pos_vel = np.clip(vel, a_min=0, a_max=None)
    power = k_spring_constant * p * pos_vel
    effort = np.trapezoid(np.clip(power, a_min=0, a_max=None), t)
    return effort

def make_successful_trial_products(T:Table = None,fig_dir = './Figure2'):
    output_directory =  f'{fig_dir}/{T.dfc}/successes_{T.dfc}'
    export_path =       f'{fig_dir}/{T.dfc}/exports'

    T.find_successful_trials()
    trials = T.df.loc[:,['Trial','success','hard_success','soft_success','pyasState']].copy()
    trials['next_trial'] = trials['Trial'].shift(-1)
    trials = trials.loc[trials['success']]
    print(trials.shape[0])

    trials = trials.loc[trials['hard_success']]
    print(trials.shape[0])

    trials['fig'] = pd.NA
    trials['v_rms'] = pd.NA
    trials['effort'] = pd.NA

    for t_num in trials.index:
        fig = Figure(figsize=(6, 6), dpi=200)
        canvas = FigureCanvas(fig)
        ax = fig.add_subplot(111)

        # success_index = -1
        # success_index = success_index+1
        # row = trials.iloc[success_index]
        row = trials.loc[t_num]
        trial1 = row['Trial']
        trial2 = row['next_trial']

        probe = []
        time = []
        for trial in [trial1,trial2]:
            pps = (trial.probe_position - trial.probeZero).squeeze()
            probe.append(pps[trial.downsample_probe])
            time.append(trial.time[trial.downsample_probe].squeeze())

        dT = np.diff([time[0][0],time[0][-1]])
        dt = np.diff([time[0][0],time[0][1]])
        time[1] = time[1]+dT+dt
        t = np.concat(time) #.ravel()
        p = np.concat(probe) #.ravel()

        start_time = trial1.as_duration
        start_idx = np.where(t>start_time)[0][0]

        # end_pos = probe[1][(time[1]>=0) & (time[1]<1)].mean()
        end_pos = (trial2.probe_position[(trial2.time>=0) & (trial2.time<1)].mean() - trial2.probeZero)
        # print(end_pos)
        end_idx = np.where(p<end_pos)[0][-1]
        end_time = t[end_idx]
        search_time = t[start_idx:end_idx]
        search = p[start_idx:end_idx]

        # v = probe_velocity(search_time,search)
        v_rms = probe_rms_velocity(search_time,search)
        search_power = probe_power(search_time,-search)
        search_effort = probe_positive_effort(search_time,-search)

        info_str = f'v_rms = {v_rms:.1f}; effort = {search_effort:.1f}'

        T.plot_some_probe_groups(T.df.loc[row.name:row.name+2,:].index,ax=ax) #,force_pos=True)
        x_lims = ax.get_xlim()
        y_lims = ax.get_ylim()
        ax.plot(x_lims,end_pos*np.array([1,1]))
        ax.plot(search_time,search)
        scaled_power = search_power/np.max((np.abs([search_power.max(),search_power.min()])))
        # ax.plot(search_time,-scaled_power)
        
        def tpos(lims,scalar=.9):
            return np.diff(lims)*scalar+lims[0]
        ax.text(tpos(x_lims,.6),tpos(y_lims),info_str)

        trials.loc[t_num,'fig'] = fig
        trials.loc[t_num,'v_rms'] = v_rms
        trials.loc[t_num,'effort'] = search_effort

    for trn in trials.index:
        trials.loc[trn,'fig'].savefig(f'{output_directory}/trial_{trn}.svg')

    cats = trials.index
    n = len(trials)
    palette = sns.color_palette("Set2", n)
    color_map = dict(zip(cats, palette))

    # Reuse proj_df and color_map from above
    export_df = trials[['success','hard_success','soft_success','pyasState','v_rms','effort']] # trial_number	Trial	success	hard_success	soft_success	pyasState	next_trial	fig	v_rms	effort
    export_df = export_df.reset_index()
    export_df['dfc'] = T.dfc
    pklname = f'{export_path}/{T.dfc}_{T.genotype}_success_df.pkl'
    print(pklname)
    export_df.to_pickle(pklname)
    
    plot_df = trials.reset_index()

    # Convert seaborn/matplotlib colors to hex for plotly
    # color_map = dict(zip(cats, palette))
    color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

    fig = px.scatter(
        plot_df,
        x="v_rms", y="effort", 
        color="pyasState",
        color_discrete_map=color_map_hex,
        hover_data=["trial_number",],
        title="v_rms vs. effort"
    )
    fig.update_traces(marker=dict(size=5, opacity=0.9))
    fig.update_layout(
        legend_title_text="trial_number",
        scene=dict(
            xaxis_title=f"v_rms",
            yaxis_title=f"effort",
        )
    )

    fig.update_layout(
        width=1100, height=850,                    # << size here
        margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
        font=dict(size=16),
        legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
        scene=dict(
            xaxis=dict(tickfont=dict(size=12)),
            yaxis=dict(tickfont=dict(size=12)),
            zaxis=dict(tickfont=dict(size=12)),
        ),
        hoverlabel=dict(font_size=14)
    )

    output_directory = f'{fig_dir}/{T.dfc}'
    os.makedirs(output_directory, exist_ok=True)

    fig.write_image(f'{output_directory}/success_effort_v_rms_{T.dfc}.png')


def fold_blocks(T:Table = None,fig_dir:str = './Figure2',norm:bool = False):
    svg_path = f'{fig_dir}/{T.dfc}'
    output_directory = f'{fig_dir}/{T.dfc}/successes_{T.dfc}'
    export_path = f'{fig_dir}/{T.dfc}/exports'

    df = T.df.loc[T.df.index>100,['pyasState','op_cnd_blocks','as_outcome','success','as_duration']]

    df = df.copy()
    df['block_pair'] = (df['op_cnd_blocks'] - 1) // 2 + 1   # pair_id 1..14

    # 2. Align trials relative to the start of each block
    df['trial_in_block'] = df.groupby('op_cnd_blocks').cumcount()
    df['probe_as'] = (df['as_duration']>0) & (df['as_outcome']=='probe')
    df = df.loc[df.trial_in_block<50]
    
    outcome_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'as_outcome'])
        .size()
        .unstack(fill_value=0)
    )
    outcome_counts = outcome_counts.loc[['hi','lo']]
    outcome_counts['max_cnts'] = outcome_counts.sum(axis=1,skipna=True)
    outcome_counts['as_off'] = outcome_counts['as_off'] + outcome_counts['as_off_late']
    outcome_counts['no_as'] = outcome_counts['no_as_no_mv'] + outcome_counts['no_as_mv']
    outcome_counts['to'] = outcome_counts['timeout_fail'] + outcome_counts['timeout']

    success_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'success'])
        .size()
        .unstack(fill_value=0)
    )
    if not True in success_counts.columns:
        success_counts[True] = 0
    success_counts = success_counts.loc[['hi','lo'],True]
    success_counts = success_counts.to_frame(name='success')
    outcome_counts['success'] = success_counts['success']

    probe_as_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'probe_as'])
        .size()
        .unstack(fill_value=0)
    )
    probe_as_counts = probe_as_counts.loc[['hi','lo']]
    if not True in probe_as_counts.columns:
        probe_as_counts[True] = 0
    probe_as_counts = probe_as_counts.loc[['hi','lo'],True]
    probe_as_counts = probe_as_counts.to_frame(name='probe_as')
    outcome_counts['probe_as'] = probe_as_counts['probe_as']
    
    outcome_counts.to_pickle(path=f'{export_path}/folded_outcome_counts_{T.dfc}.pkl')

    # del fig
    plt.close('all')
    fig = Figure(figsize=(8,8))
    lines = defaultdict(dict)

    colors = {'hi': '#1f77b4', 
            'lo': '#d62728'}
    colors = {'as_off': "#169400", 
              'no_as': "#000cad",
              'no_as_mv': "#6068cb",
              'success': "#009480",
              'to': "#FF0000",
              'probe_as': "#FF30FC",
            }

    hi_x = outcome_counts.xs('hi', level='pyasState').index
    x = {'hi': hi_x,
        'lo': outcome_counts.xs('lo', level='pyasState').index + hi_x[-1] + 1}

    outcome_to_ax = {'as_off':1,
                    'no_as':2,
                    'no_as_mv':2,
                    'success':3,
                    'to':3,
                    'probe_as':3,}

    ax_ids = sorted(set(outcome_to_ax.values()))
    ax_id_to_ax = {}
    for i, ax_id in enumerate(ax_ids, start=1):
        # nrows = len(ax_ids), ncols = 1, index = i
        ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = outcome_counts.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / outcome_counts.xs(st, level='pyasState').get('max_cnts')

            lines[outcome][st], = ax.plot(
                        x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(outcome_counts.xs(st, level='pyasState').get('max_cnts'))])

    ax_id_to_ax[1].set_ylabel('as_off')
    ax_id_to_ax[2].set_ylabel('no_as')
    ax_id_to_ax[3].set_ylabel('success')
    ax_id_to_ax[3].legend()

    os.makedirs(svg_path, exist_ok=True)
    fig.savefig(f'{svg_path}/outcomes_over_blocks_{T.dfc}.svg')
    
    display(fig)

# Export v_rms vs effort, and AS_outcomes over folded blocks

In [ ]:
# for dfc in ['250304_F3_C1','250425_F1_C1']:
for dfc in sinq.df.index:
    dfc_path = f'./Figure2/folded_blocks/{dfc}'
    success_path = f'{dfc_path}/successes_{dfc}'
    export_path =  f'{dfc_path}/exports'
    
    os.makedirs(dfc_path, exist_ok=True)
    os.makedirs(success_path, exist_ok=True)
    os.makedirs(export_path, exist_ok=True)
    
    table_chk_pnt = (not os.path.isfile(f'{dfc_path}/outcomes_over_blocks_{dfc}.svg') 
                 or not os.path.isfile(f'{export_path}/folded_outcome_counts_{dfc}.pkl') 
                 or not os.path.isfile(f'{export_path}/{dfc}_{sinq.df.loc[dfc,'genotype']}_success_df.pkl'))

    if table_chk_pnt:    
        T = sinq.restore_table(dfc)
        T.extract_trial_properties()
        T.find_successful_trials()
        make_successful_trial_products(T,fig_dir='./Figure2/folded_blocks/')
        fold_blocks(T,fig_dir='./Figure2/folded_blocks/')
    else:
        print(f'Files for {dfc} already exist')


    sinq.drop_tables()

# Plot v_rms vs. effort

In [ ]:
import glob

df_list = []

for dfc in reversed(sinq.df.index):
    export_path = os.path.join(f'./Figure2/{dfc}','exports')
    pklpat = f'{export_path}/{dfc}_*_success_df.pkl'
    file_path = glob.glob(pklpat)
    if file_path:
        file_path = file_path[0]
        print(file_path)

    df_list.append(pd.read_pickle(file_path))

success_df = pd.concat(df_list, axis=0)
cats = success_df.dfc.unique()
palette = sns.color_palette("Set2", len(cats))
color_map = dict(zip(cats, palette))
color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

fig = px.scatter(
    success_df,
    x="v_rms", y="effort", 
    color="dfc",
    color_discrete_map=color_map_hex,
    hover_data=['dfc',"trial_number",],
    title="v_rms vs. effort"
)
fig.update_traces(marker=dict(size=5, opacity=0.9))
fig.update_layout(
    legend_title_text="trial_number",
    scene=dict(
        xaxis_title=f"v_rms",
        yaxis_title=f"effort",
    )
)

fig.update_layout(
    width=1100, height=850,                    # << size here
    margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
    font=dict(size=16),
    legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
    scene=dict(
        xaxis=dict(tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=12)),
        zaxis=dict(tickfont=dict(size=12)),
    ),
    hoverlabel=dict(font_size=14)
)

# output_directory = f'./Figure2/{T.dfc}'
# os.makedirs(output_directory, exist_ok=True)

fig.write_image(f'./Figure3/success_effort_v_rms_all.png')


# Fold blocks and average outcomes across blocks

In [ ]:
op_df.loc[op_df['geno_smpl']=='w']

In [ ]:
from collections import defaultdict

for g in ['w','+;iav>kir2.1','w;iav>kir2.1','+;TH>kir2.1']:

    norm = True
    total_outcomes = None
    total_success = None
    total_probe = None
    outcome_dict = defaultdict(dict)

    g_flies =    op_df.loc[op_df['geno_smpl']==g]
    for dfc in reversed(g_flies.index):
        export_path = f'./Figure3/folded_blocks/{dfc}/exports'
        pklpat = f'{export_path}/folded_outcome_counts_{dfc}.pkl'
        file_path = glob.glob(pklpat)
        if file_path:
            file_path = file_path[0]
            print(file_path)

        outcome_dict[dfc] = pd.read_pickle(file_path)

        if total_outcomes is None:
            total_outcomes = outcome_dict[dfc].copy()
        else:
            total_outcomes = total_outcomes.add(outcome_dict[dfc], fill_value=0)

    plt.close('all')
    fig = Figure(figsize=(8,8))
    lines = defaultdict(dict)

    colors = {'hi': '#1f77b4', 
            'lo': '#d62728'}
    colors = {'as_off': "#169400", 
                'no_as': "#000cad",
                'no_as_mv': "#6068cb",
                'success': "#009480",
                'to': "#FF0000",
                'probe_as': "#FF30FC",
            }

    hi_x = total_outcomes.xs('hi', level='pyasState').index
    x = {'hi': hi_x,
        'lo': total_outcomes.xs('lo', level='pyasState').index + hi_x[-1] + 1}

    outcome_to_ax = {'as_off':1,
                    'no_as':2,
                    'no_as_mv':2,
                    'success':3,
                    'to':3,
                    'probe_as':3,}

    ax_ids = sorted(set(outcome_to_ax.values()))
    ax_id_to_ax = {}
    for i, ax_id in enumerate(ax_ids, start=1):
        ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

    # for dfc in reversed(sinq2.df.index):
    for dfc in outcome_dict.keys():
        for outcome in outcome_to_ax.keys():
            for st in ['hi','lo']:
                outcomes = outcome_dict[dfc]
                ax = ax_id_to_ax[outcome_to_ax[outcome]]
                hi_y = outcomes.xs(st, level='pyasState').get(outcome)
                if norm:
                    hi_y = hi_y / outcomes.xs(st, level='pyasState').get('max_cnts')

                # ax.scatter(
                #             x[st], hi_y, marker='.',s=5, alpha=0.9, c=colors[outcome]
                #         )
                if norm:
                    ax.set_ylim([0, 1])
                else:
                    ax.set_ylim([0, np.max(outcomes.xs(st, level='pyasState').get('max_cnts'))])


    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = total_outcomes.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / total_outcomes.xs(st, level='pyasState').get('max_cnts')

            lines[outcome][st], = ax.plot(
                        x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(total_outcomes.xs(st, level='pyasState').get('max_cnts'))])

    ax_id_to_ax[1].set_ylabel('as_off')
    ax_id_to_ax[2].set_ylabel('no_as')
    ax_id_to_ax[3].set_ylabel('success')
    ax_id_to_ax[3].legend()
    fig.savefig(f'./Figure2/outcomes_over_blocks_{_get_gstr(g)}.svg',format='svg')
    display(fig)

In [ ]:
pd.Index([i for i in range(500)])

In [ ]:
# index=
T.df.loc[T.df.index[50:450]]

In [ ]:
sinq.drop_tables()

In [ ]:
for g in ['w','+;iav>kir2.1','+;TH>kir2.1']:
    g_flies =    op_df.loc[op_df['geno_smpl']==g]
    for dfc in reversed(g_flies.index):
        T = sinq.restore_table(dfc)
        T.extract_trial_properties()
        T.fig_folder = f'./Figure3/heatmaps/{_get_gstr(g)}'
        idx=T.df.loc[T.df.index[50:450]].index
        print(f'{dfc}: making heat map for {len(idx)} trials')
        try:
            fig,ax,df = T.plot_probe_position_heatmap(index=idx,cmin=-500,cmax=10,format='svg')
        except Exception as e:
            print(f'{e}')